In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

from src.simulation.order_stream import OrderStream
from src.simulation.simulator import Simulator
from src.dynamicProgramming.dp_scheduler import DPScheduler
from src.dynamicProgramming.insertion_policy import GreedyInsertionPolicy
from src.dynamicProgramming.value_function import ValueFunction
from src.dynamicProgramming.feature_extractor import FeatureExtractor
from src.routing.route_utils import route_duration

data_path = Path("../data/features/olist/delivery_jobs_dataset.csv")

In [2]:
# loading a sample dataset
jobs = pd.read_csv(
    data_path,
    parse_dates = [
        'ready_time',
        'due_date'
    ]
)

# dropping rows without lat & lng coordinates
jobs = jobs.dropna(
    subset = [
        'pickup_lat',
        'pickup_lng',
        'delivery_lat',
        'delivery_lng'
    ]
)

# fleetsize hyperparameter
n_couriers = 2

# filtering a sample dataset
jobs = jobs.sort_values('ready_time').head(500).copy()
start = pd.Timestamp("2026-01-01 08:00:00")

jobs['ready_time'] = start + pd.to_timedelta(
    np.arange(len(jobs)) * 5,
    unit = 'm'
)

jobs['due_date'] = jobs['ready_time'] + pd.Timedelta(minutes = 30)
jobs['service_time_min'] = 5
display(jobs.head())

,job_id,order_id,seller_id,customer_id,pickup_lat,pickup_lng,delivery_lat,delivery_lng,ready_time,due_date,service_time_min,demand
30505,bfbd0f9bdef84302105ad712db648a6c,bfbd0f9bdef84302105ad712db648a6c,ecccfa2bb93b34a3bf033cc5d1dcdc69,86dc2ffce2dfff336de2f386a786e574,-25.507014,-49.275963,-20.585751,-47.863693,2026-01-01 08:00:00,2026-01-01 08:30:00,5,3.0
63623,1ff217aa612f6cd7c4255c9bfe931c8b,1ff217aa612f6cd7c4255c9bfe931c8b,4b1eaadf791bdbbad8c4a35b65236d52,b3a9bf200375f53cc5c6991919c356fd,-21.177710,-47.767820,-23.719311,-46.660397,2026-01-01 08:05:00,2026-01-01 08:35:00,5,1.0
6706,cd3b8574c82b42fc8129f6d502690c3e,cd3b8574c82b42fc8129f6d502690c3e,b499c00f28f4b7069ff6550af8c1348a,7812fcebfc5e8065d31e1bb5f0017dae,-22.600004,-47.407129,-23.032142,-45.570461,2026-01-01 08:10:00,2026-01-01 08:40:00,5,1.0
66586,ed8c7b1b3eb256c70ce0c74231e1da88,ed8c7b1b3eb256c70ce0c74231e1da88,5b179e9e8cc7ab6fd113a46ca584da81,da0ba2a9935bca5b4610b0e3bca9d3b4,-23.568771,-46.698110,-23.453962,-46.731884,2026-01-01 08:15:00,2026-01-01 08:45:00,5,1.0
87892,d207cc272675637bfed0062edffd0818,d207cc272675637bfed0062edffd0818,cca3071e3e9bb7d12640c9fbe2301306,b8cf418e97ae795672d326288dfab7a7,-21.757321,-48.829744,-22.892792,-47.173849,2026-01-01 08:20:00,2026-01-01 08:50:00,5,1.0


In [3]:
# initializing simulation components
stream = OrderStream(jobs)
policy = GreedyInsertionPolicy()

scheduler = DPScheduler(
    insertion_policy = policy,
    value_function = None,
    gamma = 0.95,
    n_couriers = n_couriers
)

simulator = Simulator(stream, scheduler)
feature_extractor = FeatureExtractor()

In [4]:
# cost function
def compute_total_cost(state):
    total_cost = 0

    for courier in state.couriers:

        # route duration penalty
        total_cost += route_duration(
            route = courier.route,
            start_time = courier.current_time
        )

        # lateness penalty
        for job in courier.completed_jobs:
            delay = max(
                0,
                (courier.current_time - job['due_date']).total_seconds() / 60
            )

            total_cost += delay * 10

    # penalty for unassigned jobs
    total_cost += len(state.active_jobs) * 30
    
    return total_cost

In [5]:
# preparing predictors and target variables for training
X, y = [], []

state = simulator.initialize(
    start_time = jobs['ready_time'].min()
)

end_time = state.current_time + pd.Timedelta(hours = 6)

while state.current_time < end_time:

    # extracting features
    features = feature_extractor.extract(state)

    # simulate one step
    simulator.step(state, step_minutes=5)

    # computing cost after simulation step
    total_cost = compute_total_cost(state)

    # safety check
    features = np.array(features, dtype = float)
    if np.any(np.isnan(features)) or np.any(np.isinf(features)):
        continue

    # skipping bad samples
    if np.isnan(total_cost) or np.isinf(total_cost):
        continue

    # restricting extreme costs
    total_cost = min(total_cost, 1000)

    # collecting data after simulation
    X.append(features)
    y.append(total_cost)


In [6]:
# training XGBoost Model
X = np.array(X)
y = np.array(y)

# shuffle dataset
idx = np.random.permutation(len(X))
X = X[idx]
y = y[idx]

# train/test split
split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# normalizing target variable
y = y / 100.0

vf = ValueFunction()
vf.fit(X_train, y_train)

print('XGB training completed!')

"""
# prediction on train set to check if training is working
for i in range(5):
    pred = vf.predict(X_train[i])
    print(f"Pred: {pred:.2f}, Actual: {y_train[i]:.2f}") """

XGB training completed!


/Users/gorkemy/Desktop/Projects/Olist Brazilian E-Commerce/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [07:28:06] WARNING: /Users/runner/work/xgboost/xgboost/src/common/error_msg.cc:56: Empty dataset at worker: 0
  bst.update(dtrain, iteration=i, fobj=obj)


'\n# prediction on train set to check if training is working\nfor i in range(5):\n    pred = vf.predict(X_train[i])\n    print(f"Pred: {pred:.2f}, Actual: {y_train[i]:.2f}") '

In [7]:
### A/B Testing - Greedy vs DP policy
# evaluation function
def run_simulation(
        df_jobs,
        policy, 
        value_function = None,
):
    scheduler = DPScheduler(
        insertion_policy = policy,
        value_function = value_function,
        gamma = 0.95,
        n_couriers = n_couriers
        )
    
    stream = OrderStream(df_jobs)
    simulator = Simulator(stream, scheduler)

    state = simulator.run(
        start_time = jobs['ready_time'].min(),
        end_time = jobs['ready_time'].min() + pd.Timedelta(hours=6),
        step_minutes=5
    )

    return state 

# evaluation metrics
def evaluation_metrics(state):
    total_completed = 0
    total_remaining = 0
    route_lengths = []
    total_service_time = 0

    for courier in state.couriers:
        total_completed += len(courier.completed_jobs)
        total_remaining += len(courier.route)
        route_lengths.append(len(courier.route))
        total_service_time += len(courier.completed_jobs) * 5

        return {
            "completed_jobs": total_completed,
            "remaining_jobs": total_remaining,
            "avg_route_length": np.mean(route_lengths),
            "total_service_time": total_service_time
        }

In [8]:
# Greedy Policy
greedy_policy = GreedyInsertionPolicy()
state_greedy = run_simulation(
    jobs,
    policy = greedy_policy,
    value_function = None
)

metrics_greedy = evaluation_metrics(state_greedy)

print("___GREEDY RESULTS___")
for k, v in metrics_greedy.items():
    print(f"{k}: {v}")

# DP Policy
dp_policy = GreedyInsertionPolicy()
state_dp = run_simulation(
    jobs,
    policy = greedy_policy,
    value_function = vf
)

metrics_dp = evaluation_metrics(state_dp)

print("___DP RESULTS___")
for k, v in metrics_dp.items():
    print(f"{k}: {v}")

___GREEDY RESULTS___
completed_jobs: 15
remaining_jobs: 0
avg_route_length: 0.0
total_service_time: 75
___DP RESULTS___
completed_jobs: 15
remaining_jobs: 0
avg_route_length: 0.0
total_service_time: 75
